In [1]:
import re
import glob
import pandas as pd
from dateutil import parser as dtparser

OUTPUT_FILE = "ft_salesforce_articles_full.csv"

all_rows = []

# This grabs NVIDIA1.txt, NVIDIA2.txt, etc.
files = sorted(glob.glob("SRM1*.txt"))

print("Files found:", files)

for file in files:
    print(f"\nProcessing {file}...")

    with open(file, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    articles = re.split(r"\nTitle:\s*", text)

    for art in articles:
        if len(art.strip()) < 100:
            continue

        def extract(pattern):
            m = re.search(pattern, art)
            return m.group(1).strip() if m else None

        title = extract(r"^(.*?)\n")
        author = extract(r"Author:\s*(.*)")
        publication = extract(r"Publication title:\s*(.*)")
        section = extract(r"Section:\s*(.*)")
        doc_type = extract(r"Document type:\s*(.*)")
        doc_id = extract(r"ProQuest document ID:\s*(.*)")
        first_page = extract(r"First page:\s*(.*)")

        # Date
        date_raw = extract(r"Publication date:\s*(.*)")
        try:
            date = dtparser.parse(date_raw).date() if date_raw else None
        except:
            date = None

        # Full text
        fulltext_match = re.search(
            r"Full text:\s*(.*?)(?:\nSubject:|\nCopyright:|\nLast updated:|\Z)",
            art,
            re.S
        )
        full_text = fulltext_match.group(1).strip() if fulltext_match else None

        all_rows.append({
            "source_file": file,
            "date": date,
            "title": title,
            "author": author,
            "publication": publication,
            "section": section,
            "first_page": first_page,
            "document_type": doc_type,
            "proquest_id": doc_id,
            "text": full_text
        })

df = pd.DataFrame(all_rows).dropna(subset=["text"])

# Remove duplicate articles across pages
df = df.drop_duplicates(subset=["proquest_id"])

df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

print(f"\nSaved {len(df)} unique articles to {OUTPUT_FILE}")


Files found: ['srm1.txt.txt']

Processing srm1.txt.txt...

Saved 99 unique articles to ft_salesforce_articles_full.csv


In [2]:
#!pip install torch transformers tqdm